# Notebook 04 — Interpretabilitate (SHAP & LIME)

**Capitolul 6 — Evaluare si interpretabilitate**

Model explicat: **RandomForest (SMOTE)** — cel mai bun dupa AUC-PR (0.8842).

Date: `X_test` real, neatins de resampling — 56.962 tranzactii, 0.172% fraude
(distributie reala). Toate rezultatele si figurile sunt reproductibile
(`random_state=42`).

Continut:
1. SHAP — `TreeExplainer` pe RandomForest (beeswarm, importanta globala, waterfall)
2. LIME — `LimeTabularExplainer` (explicatii locale pentru TP / FN / FP)
3. Celula `## SUMMARY` cu statistici cheie


## 1. Setup — incarcare date, model si predictii

In [1]:
# Librarii
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import joblib
from joblib import Parallel, delayed
from sklearn.metrics import confusion_matrix
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Cai (notebook ruleaza din notebooks/)
BASE  = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA  = os.path.join(BASE, 'data', 'preprocessed.joblib')
MODEL = os.path.join(BASE, 'models', 'rf_smote.joblib')
FIG   = os.path.join(BASE, 'figures')
os.makedirs(FIG, exist_ok=True)

plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300, 'font.size': 11})

# Incarcare date + model
data = joblib.load(DATA)
X_test  = data['X_test']
y_test  = data['y_test']
X_train = data['X_train']                 # distributie reala -> statistici LIME
feature_names = list(data['feature_names'])
model = joblib.load(MODEL)

print('X_test :', X_test.shape, '| fraude reale:', int(y_test.sum()))
print('Model  :', type(model).__name__, '| n_estimators =', model.n_estimators)
print('Features (%d): %s' % (len(feature_names), feature_names))


X_test : (56962, 30) | fraude reale: 98
Model  : RandomForestClassifier | n_estimators = 100
Features (30): ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']


In [2]:
# Predictii pe test set (prag standard 0.5)
proba = model.predict_proba(X_test)[:, 1]      # P(frauda)
pred  = (proba >= 0.5).astype(int)

tn, fp, fn_c, tp_c = confusion_matrix(y_test, pred).ravel()
print('Confusion matrix RF -> TN=%d  FP=%d  FN=%d  TP=%d' % (tn, fp, fn_c, tp_c))

idx_all = np.arange(len(y_test))
tp_mask = (y_test == 1) & (pred == 1)    # frauda detectata corect
tn_mask = (y_test == 0) & (pred == 0)    # tranzactie legitima corecta
fn_mask = (y_test == 1) & (pred == 0)    # frauda ratata
fp_mask = (y_test == 0) & (pred == 1)    # alarma falsa

# Selectie DETERMINISTICA a cazurilor de explicat
i_tp = int(idx_all[tp_mask][np.argmax(proba[tp_mask])])   # frauda cea mai sigura
i_tn = int(idx_all[tn_mask][np.argmin(proba[tn_mask])])   # legitima cea mai sigura
i_fn = int(idx_all[fn_mask][np.argmax(proba[fn_mask])])   # frauda ratata la limita
i_fp = int(idx_all[fp_mask][np.argmax(proba[fp_mask])])   # alarma falsa cea mai sigura

cases = {'TP': i_tp, 'TN': i_tn, 'FN': i_fn, 'FP': i_fp}
print('\nCazuri selectate pentru explicatii individuale:')
for k, i in cases.items():
    print('  %s: index=%d  true=%d  pred=%d  P(frauda)=%.4f'
          % (k, i, y_test[i], pred[i], proba[i]))


Confusion matrix RF -> TN=56851  FP=13  FN=15  TP=83

Cazuri selectate pentru explicatii individuale:
  TP: index=1146  true=1  pred=1  P(frauda)=1.0000
  TN: index=0  true=0  pred=0  P(frauda)=0.0000
  FN: index=12588  true=1  pred=0  P(frauda)=0.4800
  FP: index=48129  true=0  pred=1  P(frauda)=0.9700


## 2. SHAP — `TreeExplainer` pe RandomForest

In [3]:
import shap
print('SHAP version:', shap.__version__)

explainer   = shap.TreeExplainer(model)
base_values = explainer.expected_value          # [baseline_legit, baseline_frauda]
base_fraud  = float(base_values[1])
print('Valoare de baza (expected_value) clasa Frauda: %.4f' % base_fraud)

# SHAP exact pe X_test COMPLET. Arborii sunt adanci (depth ~27), deci exact
# este lent serial (~80 min). Il paralelizam pe toate nucleele (~15 min) si
# salvam in cache pentru re-rulari rapide ale figurilor.
CACHE = os.path.join(BASE, 'data', 'shap_values_rf_test.npy')

def _shap_chunk(X_chunk):
    # SHAP exact pe un sub-set de randuri (rulat in proces separat)
    return explainer.shap_values(X_chunk, check_additivity=False)

if os.path.exists(CACHE):
    shap_full = np.load(CACHE)
    method = 'cache'
    print('SHAP incarcat din cache:', shap_full.shape)
else:
    n_jobs = max(1, os.cpu_count())
    chunks = np.array_split(np.arange(len(X_test)), n_jobs)
    print('Calcul SHAP EXACT pe X_test complet (%d randuri, %d procese)...'
          % (len(X_test), n_jobs))
    t0 = time.time()
    try:
        parts = Parallel(n_jobs=n_jobs, backend='loky')(
                    delayed(_shap_chunk)(X_test[c]) for c in chunks)
        shap_full = np.concatenate(parts, axis=0)        # (n, 30, 2)
        method = 'exact-parallel'
    except Exception as ex:
        print('  Paralelizarea a esuat (%s) -> fallback aproximativ (Saabas)'
              % type(ex).__name__)
        shap_full = explainer.shap_values(X_test, approximate=True,
                                          check_additivity=False)
        method = 'approximate-saabas'
    np.save(CACHE, shap_full)
    print('Gata in %.1f min  (metoda=%s)  ->  shape %s'
          % ((time.time() - t0) / 60, method, shap_full.shape))

# Slice pentru clasa Frauda (clasa 1)
shap_fraud = shap_full[:, :, 1]                  # (n, 30)

# Verificare aditivitate: base + sum(SHAP) ~= P(frauda)
recon   = base_fraud + shap_fraud.sum(axis=1)
add_err = float(np.max(np.abs(recon - proba)))
print('shap_fraud:', shap_fraud.shape,
      '| eroare maxima aditivitate: %.2e' % add_err)

# DataFrame cu numele features pentru plot-urile SHAP
X_test_df = pd.DataFrame(X_test, columns=feature_names)


SHAP version: 0.51.0
Valoare de baza (expected_value) clasa Frauda: 0.0909
Calcul SHAP EXACT pe X_test complet (56962 randuri, 8 procese)...


Gata in 14.7 min  (metoda=exact-parallel)  ->  shape (56962, 30, 2)
shap_fraud: (56962, 30) | eroare maxima aditivitate: 9.66e-13


### 2.1 Figura 6.3 — Summary plot (beeswarm)

In [4]:
# === Figura 6.3 — SHAP beeswarm (clasa Frauda) ===
shap.summary_plot(shap_fraud, X_test_df, max_display=15, show=False)
fig = plt.gcf(); fig.set_size_inches(11, 8)
fig.axes[0].set_title('SHAP beeswarm — contributii pe clasa Frauda (RandomForest)',
                      fontsize=12)
plt.tight_layout()
out = os.path.join(FIG, 'Figura_6_3.png')
plt.savefig(out, dpi=300, bbox_inches='tight'); plt.close('all')
print('Salvat:', out)


Salvat: figures/Figura_6_3.png


### 2.2 Figura 6.4 — Bar plot (importanta globala, top 15)

In [5]:
# === Figura 6.4 — Importanta globala (|SHAP| mediu), top 15 ===
shap.summary_plot(shap_fraud, X_test_df, plot_type='bar', max_display=15, show=False)
fig = plt.gcf(); fig.set_size_inches(10, 7)
fig.axes[0].set_title('Importanta globala features (|SHAP| mediu) — top 15',
                      fontsize=12)
plt.tight_layout()
out = os.path.join(FIG, 'Figura_6_4.png')
plt.savefig(out, dpi=300, bbox_inches='tight'); plt.close('all')
print('Salvat:', out)


Salvat: figures/Figura_6_4.png


### 2.3 Tabel — Top 10 features dupa importanta SHAP medie

In [6]:
# Importanta SHAP: |SHAP| mediu (magnitudine) + SHAP semnat mediu (directie)
mean_abs    = np.abs(shap_fraud).mean(axis=0)
mean_signed = shap_fraud.mean(axis=0)
shap_imp = (pd.DataFrame({'feature': feature_names,
                          'shap_abs_mediu': mean_abs,
                          'shap_semnat_mediu': mean_signed})
            .sort_values('shap_abs_mediu', ascending=False)
            .reset_index(drop=True))
top10 = shap_imp.head(10).copy()
print('TOP 10 features dupa |SHAP| mediu (clasa Frauda):')
print(top10.to_string(index=False,
      formatters={'shap_abs_mediu': '{:.5f}'.format,
                  'shap_semnat_mediu': '{:+.5f}'.format}))


TOP 10 features dupa |SHAP| mediu (clasa Frauda):
feature shap_abs_mediu shap_semnat_mediu
    V14        0.01990          -0.01766
    V12        0.01385          -0.01267
     V4        0.01230          -0.00704
    V10        0.01066          -0.00925
    V17        0.00821          -0.00719
    V11        0.00727          -0.00645
     V3        0.00698          -0.00593
    V16        0.00487          -0.00387
     V7        0.00256          -0.00185
     V2        0.00217          -0.00154


### 2.4 Figura 6.5 — Waterfall: o frauda corecta (TP) si o legitima corecta (TN)

In [7]:
# === Figura 6.5 — Waterfall pentru un TP si un TN ===
# shap.plots.waterfall deseneaza pe figura curenta -> randam fiecare separat
# si le combinam side-by-side.
def _render_waterfall(i, tmp_path, max_display=12):
    e = shap.Explanation(values=shap_fraud[i], base_values=base_fraud,
                         data=X_test[i], feature_names=feature_names)
    plt.figure()
    shap.plots.waterfall(e, max_display=max_display, show=False)
    plt.gcf().set_size_inches(9, 6.5)
    plt.savefig(tmp_path, dpi=200, bbox_inches='tight'); plt.close('all')

tmp_tp = os.path.join(FIG, '_tmp_wf_tp.png')
tmp_tn = os.path.join(FIG, '_tmp_wf_tn.png')
_render_waterfall(i_tp, tmp_tp)
_render_waterfall(i_tn, tmp_tn)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
axes[0].imshow(plt.imread(tmp_tp)); axes[0].axis('off')
axes[0].set_title('Frauda detectata corect (TP) - idx %d, P(frauda)=%.2f'
                  % (i_tp, proba[i_tp]), fontsize=12)
axes[1].imshow(plt.imread(tmp_tn)); axes[1].axis('off')
axes[1].set_title('Tranzactie legitima corecta (TN) - idx %d, P(frauda)=%.2f'
                  % (i_tn, proba[i_tn]), fontsize=12)
plt.tight_layout()
out = os.path.join(FIG, 'Figura_6_5.png')
plt.savefig(out, dpi=300, bbox_inches='tight'); plt.close('all')
os.remove(tmp_tp); os.remove(tmp_tn)
print('Salvat:', out)


Salvat: figures/Figura_6_5.png


## 3. LIME — `LimeTabularExplainer`

In [8]:
from lime.lime_tabular import LimeTabularExplainer

# Statisticile features se invata pe setul de antrenare (distributie reala)
lime_expl = LimeTabularExplainer(
    X_train, feature_names=feature_names,
    class_names=['Legitima', 'Frauda'], mode='classification',
    discretize_continuous=True, random_state=RANDOM_STATE)

# 3 predictii individuale: 1 TP, 1 FN, 1 FP
lime_cases = [('TP - Frauda detectata', i_tp),
              ('FN - Frauda ratata',    i_fn),
              ('FP - Alarma falsa',     i_fp)]

lime_results = {}
tmp_files = []
for label, i in lime_cases:
    exp = lime_expl.explain_instance(X_test[i], model.predict_proba,
                                     num_features=10, num_samples=5000)
    lime_results[label] = exp.as_list(label=1)          # contributii clasa Frauda
    f = exp.as_pyplot_figure(label=1); f.set_size_inches(6.5, 5)
    tmp = os.path.join(FIG, '_tmp_lime_%d.png' % i)
    f.savefig(tmp, dpi=200, bbox_inches='tight'); plt.close(f)
    tmp_files.append(tmp)
    print('LIME explicat: %s (idx %d)' % (label, i))


LIME explicat: TP - Frauda detectata (idx 1146)


LIME explicat: FN - Frauda ratata (idx 12588)


LIME explicat: FP - Alarma falsa (idx 48129)


### 3.1 Figura 6.6 — LIME side-by-side (TP / FN / FP)

In [9]:
# === Figura 6.6 — LIME pentru cele 3 cazuri, side-by-side ===
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
for ax, (label, i), tmp in zip(axes, lime_cases, tmp_files):
    ax.imshow(plt.imread(tmp)); ax.axis('off')
    ax.set_title('%s\nidx %d, P(frauda)=%.2f' % (label, i, proba[i]), fontsize=12)
plt.tight_layout()
out = os.path.join(FIG, 'Figura_6_6.png')
plt.savefig(out, dpi=300, bbox_inches='tight'); plt.close('all')
for t in tmp_files:
    os.remove(t)
print('Salvat:', out)

# Detaliu numeric LIME (semn + => pro-frauda, - => pro-legitim)
for label in lime_results:
    print('\n%s:' % label)
    for feat, w in lime_results[label]:
        print('   %+.4f  %s' % (w, feat))


Salvat: figures/Figura_6_6.png

TP - Frauda detectata:
   +0.0400  V4 > 0.74
   +0.0177  V14 <= -0.43
   +0.0123  V12 <= -0.40
   +0.0118  V17 <= -0.48
   +0.0115  V20 <= -0.21
   +0.0109  V13 <= -0.65
   -0.0108  V1 <= -0.92
   +0.0105  V8 <= -0.21
   +0.0093  V10 <= -0.54
   +0.0073  V2 > 0.80

FN - Frauda ratata:
   +0.0364  V4 > 0.74
   +0.0175  V14 <= -0.43
   -0.0116  V1 <= -0.92
   +0.0103  V17 <= -0.48
   +0.0093  V8 <= -0.21
   +0.0089  V10 <= -0.54
   +0.0080  V12 <= -0.40
   +0.0061  V26 <= -0.33
   +0.0058  V2 > 0.80
   +0.0056  V11 > 0.74

FP - Alarma falsa:
   +0.0368  V4 > 0.74
   +0.0185  V14 <= -0.43
   +0.0109  V17 <= -0.48
   +0.0109  V10 <= -0.54
   -0.0094  V1 <= -0.92
   +0.0073  V2 > 0.80
   +0.0071  V12 <= -0.40
   +0.0062  V11 > 0.74
   +0.0059  V3 <= -0.89
   +0.0043  V16 <= -0.47


## SUMMARY

In [10]:
# ============================================================
# SUMMARY — Interpretabilitate (SHAP + LIME)
# ============================================================
print('=' * 60)
print('SUMMARY - Capitolul 6: Interpretabilitate (SHAP + LIME)')
print('=' * 60)
print('Model    : RandomForest (SMOTE), AUC-PR=0.8842')
print('X_test   : %d tranzactii | fraude reale: %d (%.3f%%)'
      % (len(y_test), int(y_test.sum()), 100 * y_test.mean()))
print('Conf.matr: TN=%d  FP=%d  FN=%d  TP=%d' % (tn, fp, fn_c, tp_c))
print('SHAP     : metoda=%s | base(frauda)=%.4f | err.aditivitate=%.2e'
      % (method, base_fraud, add_err))

print('\n--- TOP 10 FEATURES dupa |SHAP| mediu ---')
print(top10.to_string(index=False,
      formatters={'shap_abs_mediu': '{:.5f}'.format,
                  'shap_semnat_mediu': '{:+.5f}'.format}))

# Directia: features care imping spre FRAUDA vs spre LEGITIM
fraud_rows = (y_test == 1)
legit_rows = (y_test == 0)
df_dir = pd.DataFrame({
    'feature': feature_names,
    'medie_pe_FRAUDE':   shap_fraud[fraud_rows].mean(axis=0),
    'medie_pe_LEGITIME': shap_fraud[legit_rows].mean(axis=0)})
top_push_fraud = df_dir.sort_values('medie_pe_FRAUDE', ascending=False).head(8)
top_push_legit = df_dir.sort_values('medie_pe_LEGITIME').head(8)

print('\n--- Features care IMPING spre FRAUDA (medie SHAP pe tranzactiile frauda) ---')
print(top_push_fraud.to_string(index=False,
      formatters={'medie_pe_FRAUDE': '{:+.5f}'.format,
                  'medie_pe_LEGITIME': '{:+.5f}'.format}))
print('\n--- Features care IMPING spre LEGITIM (medie SHAP cea mai negativa) ---')
print(top_push_legit.to_string(index=False,
      formatters={'medie_pe_FRAUDE': '{:+.5f}'.format,
                  'medie_pe_LEGITIME': '{:+.5f}'.format}))

print('\n--- INDEX-URI EXACTE ale cazurilor explicate ---')
for k, i in cases.items():
    print('  %s: index=%-6d true=%d pred=%d P(frauda)=%.4f'
          % (k, i, y_test[i], pred[i], proba[i]))

print('\n--- LIME: rezumat contributii (top 3 pe directie) ---')
for label in lime_results:
    pos = [f for f, w in lime_results[label] if w > 0][:3]
    neg = [f for f, w in lime_results[label] if w < 0][:3]
    print('  %s' % label)
    print('     pro-frauda : %s' % pos)
    print('     pro-legitim: %s' % neg)

print('\n--- FIGURI GENERATE (figures/, 300 dpi) ---')
for f in ['Figura_6_3.png', 'Figura_6_4.png', 'Figura_6_5.png', 'Figura_6_6.png']:
    p = os.path.join(FIG, f)
    print('  %s  (%s)' % (f, 'OK' if os.path.exists(p) else 'LIPSA'))


SUMMARY - Capitolul 6: Interpretabilitate (SHAP + LIME)
Model    : RandomForest (SMOTE), AUC-PR=0.8842
X_test   : 56962 tranzactii | fraude reale: 98 (0.172%)
Conf.matr: TN=56851  FP=13  FN=15  TP=83
SHAP     : metoda=exact-parallel | base(frauda)=0.0909 | err.aditivitate=9.66e-13

--- TOP 10 FEATURES dupa |SHAP| mediu ---
feature shap_abs_mediu shap_semnat_mediu
    V14        0.01990          -0.01766
    V12        0.01385          -0.01267
     V4        0.01230          -0.00704
    V10        0.01066          -0.00925
    V17        0.00821          -0.00719
    V11        0.00727          -0.00645
     V3        0.00698          -0.00593
    V16        0.00487          -0.00387
     V7        0.00256          -0.00185
     V2        0.00217          -0.00154

--- Features care IMPING spre FRAUDA (medie SHAP pe tranzactiile frauda) ---
feature medie_pe_FRAUDE medie_pe_LEGITIME
    V14        +0.19224          -0.01802
    V17        +0.15073          -0.00746
    V10        +0.11